In [4]:
import math
import os
from pathlib import Path

import numpy as np
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import random_split
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet34


In [5]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

In [6]:
## routine to print stats of a model
def print_model_weight_stats(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params

    print("{}: Number of fronzen param = {:,d}, number of trainable param = {:,}".format(type(model), frozen_params,
                                                                                         trainable_params))


## calculate the pareto value
def compute_pareto_value(acc, msize):
    ## accuracy must be between 0.0 and 0.9999
    ## model size should be in millions, between 1.0 and 900!
    if not (0.0 <= acc <= 0.9999):
        raise ValueError("acc range must be between 0.0 and 0.9999")

    if not (1.0 <= msize <= 900):
        raise ValueError("model sizse must be in millions, between 1.0M and 900M")

    xdist = 0.99 - acc
    ydist = math.log10(np.float64(msize))

    # assumption: range of accuracy is between 0.99 to 0.80 -> delta 0.2
    # range of model is from 1M to 100M -> delta 2 after taking log
    # hence scale model by 0.1
    ydist = 0.1 * ydist

    z_score = math.sqrt(xdist * xdist + ydist * ydist)
    return (z_score)


## use GPU if available
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

In [7]:
# ==== CONFIG ====
DATA_ROOT = Path("data/caltech101")  # <-- CHANGE THIS
BATCH_SIZE = 32
NUM_EPOCHS = 10
LR = 1e-3
VAL_SPLIT = 0.2
NUM_WORKERS = 4

In [8]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])


In [9]:
full_dataset = datasets.ImageFolder(root=os.path.join(DATA_ROOT, '101_ObjectCategories'),
                                    transform=train_transform)

## ✅ Step 1: Sanity-check the dataset (DO NOT SKIP)

In [10]:
print("Total images", len(full_dataset))
print("Number of classes", len(full_dataset.classes))
print("First 10 classes", len(full_dataset.classes[0]))


Total images 9144
Number of classes 102
First 10 classes 17


## ✅ Step 2: Train/Validation Split

In [11]:
val_ratio = 0.2


In [12]:
n_val = int(len(full_dataset) * 0.8)
n_train = len(full_dataset) - n_val

train_dataset, val_dataset = random_split(full_dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42))


In [13]:
val_dataset.dataset.transform = val_transform

## ✅ Step 4: Create DataLoaders

In [14]:
train_loader = DataLoader(train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True, 
                          num_workers=4,
                          pin_memory=True)

val_loader = DataLoader(val_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True, 
                          num_workers=4,
                          pin_memory=True)

 ## ✅ Step 5: Load pretrained model (transfer learning)
 ResNet 34

In [15]:
model = resnet34(pretrained = True)

/home/mehrdad/Main/ODU/ODU Courses/Deep Learning/pythonProject2/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/mehrdad/Main/ODU/ODU Courses/Deep Learning/pythonProject2/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


## ✅ Step 6: Freeze backbone

In [16]:
for parameter in model.parameters():
    parameter.requires_grad = False

## ✅ Step 7: Replace classifier head

In [17]:
num_features = model.fc.in_features
num_classes = len(full_dataset.classes)

model.fc = torch.nn.Linear (num_features, num_classes)



## ✅ Step 8: Move model to device

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

## ✅ Step 9: Loss function & optimizer

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(),
                       lr=LR)

## ✅ Step 10: Training loop

In [23]:
def train_one_epoch(model, loader):
    model.train()
    running_loss = 0.0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss +=loss.item()
    return running_loss / len(loader)

## ✅ Step 11: Validation loop

In [24]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


In [ ]:
epochs = 10

for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader)
    val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")
